In [1]:
import logging
import os
import sys
import traceback
import yaml

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

from hydra import initialize, compose
from omegaconf import DictConfig
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, "../../shared_utils")
from experiment_utils import get_forward_model
from sc_exp_design.constants import DataFields

import torch.nn.functional as F

## Initialize recipes to predict for 

In [2]:
order = [2.5, 15, 25, 50, 75, 100]

recipes_to_predict = np.array([
    # [10.00, 2.5, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 0],
    [10.00, 2.5, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 0],
    [10.00, 15, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 0],
    [10.00, 25.00, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 0],
    [10.00, 50.00, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 0],
    [10.00, 75.00, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 0],
    [10.00, 100.00, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 0]])
recipes_to_predict = np.log1p(recipes_to_predict)

recipes_to_predict_t = torch.from_numpy(recipes_to_predict).unsqueeze(1).repeat(1, 1000, 1)

# Collect ground truths 

In [3]:
adata_l1 = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop0/adata_500k.h5ad")
adata_l4 = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop4/replicate/processed/adata_loop4_500k_residual_processed.h5ad")
adata_l4p5 = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop4/replicate/processed/adata_loop4p5_500k_residual_processed.h5ad")

In [4]:
cell_types = np.unique(adata_l4.obs.cell_type)

In [5]:
gt_l4 = adata_l4.obs.cell_type.value_counts() / adata_l4.shape[0]
gt_l4p5 = adata_l4p5.obs.cell_type.value_counts() / adata_l4p5.shape[0]

# Load loop3 fwd model 

In [6]:
with initialize(config_path="../../inverse/loss_guidance/config", version_base=None):
    config = compose(config_name="run_inverse",
                     overrides=[
                        "paths=loop4_replicate",
                        "annotation=bloodplus_loop3"
                    ])

ANNOTATION_CFG_FILE = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/annotation/default.yaml"
with open(ANNOTATION_CFG_FILE, "r") as fb:
    annotation_cfg = yaml.safe_load(fb)
protocol_cols = annotation_cfg["protocol_columns"]

In [ ]:
logger = logging.getLogger(__name__)

forward_model, (
    flow_matching,
    target_prediction_model
) = get_forward_model(config, logger=logger)

In [ ]:
train_adata_g = target_prediction_model.train_data.adata

# Prepare label encoder
ct_le = LabelEncoder()
ct_values = train_adata_g.obs["cell_type"].values
ct_le.fit(ct_values)
classes = np.array(ct_le.classes_.tolist())

In [ ]:
# Initialize the batch 
condition_repr = next(iter(forward_model.forward_model.train_data.data.perturbation_covariates))
recipes_to_predict_t = recipes_to_predict_t.to(forward_model.device).float()
batch_dict = {DataFields.PERTURBATION_DATA: {condition_repr: recipes_to_predict_t}}

# Save results
y_preds = []

for _ in tqdm(range(5)):
    forward_out = forward_model.predict(
        batch_dict,
        no_grad=True,
        fix_noise=False,
        num_time_steps=100
    )
    # Collect predictions 
    y_pred = forward_out["target_prediction_data"]["cell_type"].view(recipes_to_predict_t.shape[0], 
                                                                     recipes_to_predict_t.shape[1], 
                                                                     len(classes))  
    y_pred_softmax = F.softmax(y_pred, dim=-1).detach().cpu().numpy()

    y_pred = y_pred_softmax.argmax(-1)
    y_preds.append(y_pred)

In [ ]:
for i in y_preds:
    print(i.shape)

In [ ]:
y_preds = np.stack(y_preds, axis=0)

In [ ]:
pred_classes_loop3 = classes[y_preds]

# Plateau plot 

In [ ]:
to_plot_plateau = {"TPO": [], 
                   "Mgk_Ery_proportion": []}

mgkpro_ery_lineage = ["MgKEryPro1", "MgKEryPro2", "EryPro", "late_MgkPro"]

In [ ]:
for rep in range(len(pred_classes_loop3)):
    for (i, tpo_val) in enumerate(order):
        to_plot_plateau["TPO"].append(tpo_val)
        value_counts = pd.DataFrame(pred_classes_loop3[rep, i]).value_counts() / pred_classes_loop3[rep, i].shape[0]
        for ct in cell_types:
            if ct not in value_counts.index:
                value_counts[ct] = 0
        value_counts = value_counts.loc[mgkpro_ery_lineage]
        to_plot_plateau["Mgk_Ery_proportion"].append(value_counts.sum())

In [ ]:
to_plot_plateau_df = pd.DataFrame(to_plot_plateau)

# Compare to real data in a plot 

In [ ]:
real_2p5 = 205
real_25 = "259"
real_100 = "260"

In [ ]:
adata_2p5 =  adata_l1[adata_l1.obs.experiment_number==real_2p5] 
prop_2p5 = adata_2p5.obs.cell_type.value_counts() / adata_2p5.obs.cell_type.value_counts().sum()
prop_2p5 = prop_2p5[mgkpro_ery_lineage].sum()

adata_25 =  adata_l4[adata_l4.obs.experiment_number==real_25] 
prop_25 = adata_25.obs.cell_type.value_counts() / adata_25.obs.cell_type.value_counts().sum()
prop_25 = prop_25[mgkpro_ery_lineage].sum()

adata_100 =  adata_l4p5[adata_l4p5.obs.experiment_number==real_100] 
prop_100 = adata_100.obs.cell_type.value_counts() / adata_100.obs.cell_type.value_counts().sum()
prop_100 = prop_100[mgkpro_ery_lineage].sum()

In [ ]:
sns.set_style("white")  # no grid, clean background
fig, ax = plt.subplots(figsize=(3, 3))
sns.lineplot(
    data=to_plot_plateau_df,
    x="TPO",
    y="Mgk_Ery_proportion",
    marker="^",
    markersize=11,
    linewidth=1.5,
    color="dimgray",
    ax=ax
)
# Overlay the real (measured) proportions as red squares
real_tpo = [2.5, 25, 100]
real_props = [prop_2p5, prop_25, prop_100]
ax.scatter(
    real_tpo,
    real_props,
    marker="s",
    s=80,
    color="firebrick",
    zorder=5,
    label="Real proportion"
)
ax.set_xlabel("TPO", fontsize=12)
ax.set_ylabel("Mgk_Ery proportion", fontsize=12)
# ax.set_ylim(0, 0.4)

# Explicitly set x-ticks to include 2.5
ax.set_xticks([2.5, 25, 50, 75, 100])
ax.set_xticklabels(["2.5", "25", "50", "75", "100"])

# Clean up spines
sns.despine(ax=ax)
plt.savefig("mgk_ery_proportion.svg", format="svg", bbox_inches="tight")
plt.show()